In [1]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from pythainlp.tokenize import word_tokenize # pip install pythainlp
import torch


/Users/master_slave/python_project/poc-privacy-preserving-voc-data/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
name="Porameht/wangchanberta-thainer-corpus-v2-2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForTokenClassification.from_pretrained(name)

sentence="นายปรเมศ คุ้มสมบัติ 552/44 หมู่ 1 บ้านหนองบัว ต.ภูหอ อ.ภูหลวง จ.เลย 42230"
cut=word_tokenize(sentence.replace(" ", "<_>"))
inputs=tokenizer(cut,is_split_into_words=True,return_tensors="pt")

ids = inputs["input_ids"]
mask = inputs["attention_mask"]
# forward pass
outputs = model(ids, attention_mask=mask)
logits = outputs[0]

predictions = torch.argmax(logits, dim=2)
predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]


In [3]:
def fix_span_error(words,ner):
    _ner = []
    _ner=ner
    _new_tag=[]
    for i,j in zip(words,_ner):
        #print(i,j)
        i=tokenizer.decode(i)
        if i.isspace() and j.startswith("B-"):
            j="O"
        if i=='' or i=='<s>' or i=='</s>':
            continue
        if i=="<_>":
            i=" "
        _new_tag.append((i,j))
    return _new_tag

ner_tag=fix_span_error(inputs['input_ids'][0],predicted_token_class)

In [4]:
ner_tag

[('นาย', 'B-PERSON'),
 ('ปร', 'I-PERSON'),
 ('เม', 'I-PERSON'),
 ('ศ', 'I-PERSON'),
 (' ', 'B-LOCATION'),
 ('คุ้ม', 'I-PERSON'),
 ('สมบัติ', 'I-PERSON'),
 (' ', 'O'),
 ('55', 'O'),
 ('2/', 'O'),
 ('44', 'O'),
 (' ', 'B-LOCATION'),
 ('หมู่', 'B-LOCATION'),
 (' ', 'I-LOCATION'),
 ('1', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('บ้าน', 'B-LOCATION'),
 ('หนอง', 'I-LOCATION'),
 ('บัว', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('ต', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('ภู', 'I-LOCATION'),
 ('หอ', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('อ', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('ภู', 'I-LOCATION'),
 ('หลวง', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('จ', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('เลย', 'I-LOCATION'),
 (' ', 'B-ZIP'),
 ('4', 'B-ZIP'),
 ('22', 'B-ZIP'),
 ('30', 'B-ZIP')]

In [5]:
from tabulate import tabulate
# Display the NER tags in a table format   
print(tabulate(ner_tag, headers=["Token", "NER Tag"], tablefmt="grid"))

+---------+------------+
| Token   | NER Tag    |
+=========+============+
| นาย     | B-PERSON   |
+---------+------------+
| ปร      | I-PERSON   |
+---------+------------+
| เม      | I-PERSON   |
+---------+------------+
| ศ       | I-PERSON   |
+---------+------------+
|         | B-LOCATION |
+---------+------------+
| คุ้ม      | I-PERSON   |
+---------+------------+
| สมบัติ    | I-PERSON   |
+---------+------------+
|         | O          |
+---------+------------+
| 55      | O          |
+---------+------------+
| 2/      | O          |
+---------+------------+
| 44      | O          |
+---------+------------+
|         | B-LOCATION |
+---------+------------+
| หมู่      | B-LOCATION |
+---------+------------+
|         | I-LOCATION |
+---------+------------+
| 1       | I-LOCATION |
+---------+------------+
|         | B-LOCATION |
+---------+------------+
| บ้าน     | B-LOCATION |
+---------+------------+
| หนอง    | I-LOCATION |
+---------+------------+
| บัว      | I-LOC

In [6]:
from termcolor import colored

def highlight_ner(ner_tag):
    out = ""
    current_tag = None
    for token, tag in ner_tag:
        if tag == "O":
            out += token
            current_tag = None
        else:
            entity = tag.split("-")[-1]
            if tag.startswith("B-") or current_tag != entity:
                out += f"[{entity}:" + colored(token, "cyan")  # แสดงด้วยสี
            elif tag.startswith("I-") and current_tag == entity:
                out += colored(token, "cyan")
            current_tag = entity
    if current_tag:
        out += "]"
    return out

print(highlight_ner(ner_tag))


[PERSON:นายปรเมศ[LOCATION: [PERSON:คุ้มสมบัติ 552/44[LOCATION: [LOCATION:หมู่ 1[LOCATION: [LOCATION:บ้านหนองบัว[LOCATION: [LOCATION:ต.ภูหอ[LOCATION: [LOCATION:อ.ภูหลวง[LOCATION: [LOCATION:จ.เลย[ZIP: [ZIP:4[ZIP:22[ZIP:30]


In [7]:
# Reconstruct structured PII

from collections import defaultdict

def extract_entities(ner_tag):
    result = defaultdict(str)
    current_entity = ""
    current_type = None
    for word, tag in ner_tag:
        if tag.startswith("B-"):
            current_type = tag[2:]
            result[current_type] += word
        elif tag.startswith("I-") and current_type:
            result[current_type] += word
        else:
            current_type = None
    return dict(result)

print(extract_entities(ner_tag))


{'PERSON': 'นายปรเมศ', 'LOCATION': ' คุ้มสมบัติ หมู่ 1 บ้านหนองบัว ต.ภูหอ อ.ภูหลวง จ.เลย', 'ZIP': ' 42230'}


In [3]:
text = "แนะนำ กฟอ.สวี จังหวัดชุมพร  เรื่องไฟฟ้าตกบ่อย บ้านเลขที่ 253 ม.7 บ.น้ำชล ต.ทุ่งระยะ อ.สวี จ.ชุมพร ผู้ใช้ไฟฟ้าแจ้งว่าในพื้นที่มีกระแสไฟฟ้าตกบ่อย ซึ่งทางผู้ใช้ไฟฟ้าแจ้งว่าได้เคยมีการแนะนำเข้าไปแล้วเมื่อปี 2566 แต่ปัจจุบันก็ยังมีไฟฟ้าตกในพื้นที่ ผู้ใช้ไฟแจ้งว่าบ้านอยู่ห่างจากหม้อแปลงไม่เกิน 300 เมตร แต่ไฟมาไม่ถึง 200 วัตต์ จากเหตุการณ์ที่เกิดขึ้นเกรงว่าอุปกรณ์เครื่องใช้ไฟฟ้าจะได้รับความเสียหาย จึงต้องการให้ กฟอ.สวี และหน่วยงานที่เกี่ยวข้องเข้าดำเนินการตรวจสอบปรับปรุงแก้ไขระบบคุณภาพไฟฟ้าในพื้นที่ให้ดียิ่งขึ้น//ผู้ใช้ไฟฟ้าคุณปิยะวัฒน์  หมายเลขติดต่อกลับ 0840807317"

In [22]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from pythainlp.tokenize import word_tokenize
from openai import OpenAI
import re
client = OpenAI(api_key="OPENAI_API_KEY_REDACTED")

# โหลด tokenizer และ model
model_name = "Porameht/wangchanberta-thainer-corpus-v2-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
label_list = model.config.id2label

def ner_highlight(text):
    # inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    cut = word_tokenize(text.replace(" ", "<_>"))
    inputs = tokenizer(cut, is_split_into_words=True, return_tensors="pt")
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    outputs = model(ids, attention_mask=mask)
    logits = outputs[0]
    predictions = torch.argmax(logits, dim=2)
    predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
    
    def fix_span_error(words, ner):
        _ner_tag = []
        for i, j in zip(words, ner):
            i = tokenizer.decode(i)
            if i.isspace() and j.startswith("B-"):
                j = "O"
            if i == '' or i == '<s>' or i == '</s>':
                continue
            if i == "<_>":
                i = " "
            _ner_tag.append((i, j))
        return _ner_tag
    ner_tag = fix_span_error(inputs['input_ids'][0], predicted_token_class)

    html = ""
    for token, label in ner_tag:
        if token in tokenizer.all_special_tokens:
            continue
        word = token.replace("▁", " ")
        if label != "O":
            label_type = label.split("-")[-1]
            html += f"<mark style='background-color: #d1e7dd;' title='{label_type}'>{word}</mark>"
        else:
            html += word
    return html.strip()

def ner_mask(text):
    cut = word_tokenize(text.replace(" ", "<_>"))
    inputs = tokenizer(cut, is_split_into_words=True, return_tensors="pt")
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    outputs = model(ids, attention_mask=mask)
    logits = outputs[0]
    predictions = torch.argmax(logits, dim=2)
    predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
    
    def fix_span_error(words, ner):
        _ner_tag = []
        for i, j in zip(words, ner):
            i = tokenizer.decode(i)
            if i.isspace() and j.startswith("B-"):
                j = "O"
            if i == '' or i == '<s>' or i == '</s>':
                continue
            if i == "<_>":
                i = " "
            _ner_tag.append((i, j))
        return _ner_tag
    
    ner_tag = fix_span_error(inputs['input_ids'][0], predicted_token_class)
    
    masked = ""
    current_entity = None 
    
    for token, tag in ner_tag:
        if tag == "O":
            masked += token 
            current_entity = None
        else:
            entity = tag.split('-')[1]
            if tag.startswith('B-') or current_entity != entity:
                masked += f"[{entity}]"
                current_entity = entity
            elif tag.startswith('I-') and current_entity == entity:
                continue
            else:
                current_entity = None
                masked += token
    return re.sub(r'(\[[A-Z]+\])\1+', r'\1', masked.strip())


def summarize_issue(masked_text):
    system_prompt = """คุณคือผู้ช่วยวิเคราะห์ VOC ที่เน้นหาประเด็นปัญหาเชิงเนื้อหาแบบกระชับและเข้าใจง่ายในโทนภาษาที่เป็นกันเองเหมือนพูดกับผู้บริหาร"""
    user_prompt = f"""จากข้อความที่ผ่านการปกปิดข้อมูลส่วนบุคคลดังนี้:\n\n{masked_text}\n\nช่วยสรุปประเด็นปัญหาหลักที่เกิดขึ้นให้เข้าใจง่าย"""

    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()

def summarize_topic(masked_text):
    system_prompt = """คุณคือผู้ช่วยวิเคราะห์ VOC ที่สามารถจำแนกประเภทเสียงของลูกค้าได้ตามหมวดหมู่ต่อไปนี้:
1. แจ้งปัญหาด้านบริการ
2. แจ้งปัญหาคุณภาพไฟฟ้า
3. ชื่นชม
4. ร้องขอ
5. แจ้งเบาะแส

หากข้อความเข้าข่ายมากกว่าหนึ่งประเภทแบบมีความน่าจะเป็นสูงมากๆ ให้ระบุไม่เกิน 2 ประเภท และให้ตอบกลับโดย:
- ระบุประเภทเสียงของลูกค้าเป็นชื่อหมวดหมู่ (ตามรายการด้านบน)
- ตามด้วยเหตุผลสั้นๆ แบบเข้าใจง่ายในรูปแบบ: “เพราะ...”

ตอบให้กระชับ ชัดเจน และเป็นกันเองเหมือนพูดกับผู้บริหาร"""

    user_prompt = f"""จากข้อความที่ผ่านการปกปิดข้อมูลส่วนบุคคลดังนี้:\n\n{masked_text}\n\nช่วยจำแนกประเภทของเสียงของลูกค้า พร้อมเหตุผลแบบกระชับ"""

    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

def sentiment_score(summary_text):
    user_prompt = f"""จากข้อความนี้:\n\n{summary_text}\n\nช่วยให้คะแนนความพึงพอใจในช่วง -5 (ไม่พอใจอย่างรุนแรง) ถึง +5 (พึงพอใจมาก) พร้อมสรุปสั้น ๆ ว่าทำไมถึงได้คะแนนนั้น"""

    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "system", "content": "คุณคือผู้ช่วยให้คะแนนความพึงพอใจจากข้อความร้องเรียน"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=512
    )
    return response.choices[0].message.content.strip()

def process(text):
    highlighted = ner_highlight(text)
    masked = ner_mask(text)
    summary = summarize_issue(masked)
    voc_type = summarize_topic(masked)
    scorecard = sentiment_score(summary)
    return highlighted, masked, summary, voc_type, scorecard

with gr.Blocks(css="""
.full-height textarea {
    height: auto !important;
    min-height: 200px;
}
""") as app:
    gr.Markdown("""
    # 🔒 Thai PII Anonymizer with NER (Custom Token Classification)
    ใช้ PyThaiNLP สำหรับตัดคำภาษาไทย และโมเดล NER แบบ custom เพื่อปกปิดข้อมูลส่วนบุคคล และสรุปประเด็นปัญหาเชิงเนื้อหา พร้อมวิเคราะห์ความพึงพอใจ
    """)

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(lines=5, label="Input Text", placeholder="ใส่ข้อความที่ต้องการประมวลผล...")
            submit_btn = gr.Button("Submit", variant="primary")
        with gr.Column():
            voice_type_output = gr.Textbox(label="ประเภทเสียงของลูกค้า", lines=2, elem_classes="full-height")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### NER Result (Highlighted)")
            highlight_output = gr.HTML()
        with gr.Column():
            mask_output = gr.Textbox(label="Anonymized Output (Masking)", lines=10, elem_classes="full-height")

    gr.Markdown("### 📌 สรุปประเด็นปัญหาเชิงเนื้อหา")
    summary_output = gr.Textbox(lines=6, elem_classes="full-height")

    gr.Markdown("### 🧭 คะแนนความพึงพอใจจากข้อความ")
    sentiment_output = gr.Textbox(lines=3, elem_classes="full-height")

    submit_btn.click(fn=process, inputs=input_text, outputs=[highlight_output, mask_output, summary_output, voice_type_output, sentiment_output])

app.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


นายทองอินทร์ ผงทอง บ้านเลขที่ 167 หมู่ 13 ต.สระนกแก้ว อ.โพนทอง จ.ร้อยเอ็ด เบอร์โทร 0986451488

In [23]:
app.close()

Closing server running on port: 7860


[ORGANIZATION][ORGANIZATION][LOCATION][LOCATION] เรื่องสถานะการขอใช้ไฟฟ้าเครื่องที่ 2 ของบ้านเลขที่ 45/263[LOCATION])[LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION] หมายเลขผู้ใช้ไฟฟ้า[PHONE][PHONE]89 ชื่อผู้จดทะเบียนใช้ไฟฟ้า[PERSON]ย์ วันที่ยื่นเรื่อง[DATE][DATE]67 มิเตอร์ขนาด 14 (45) แอมป์ ผู้ใช้ไฟฟ้าแจ้งว่าวันที่[DATE][DATE]67 ได้ทําการติดต่อการไฟฟ้า ทางการไฟฟ้าแจ้งว่าจะเข้าตรวจสอบวันถัดไปเป็นวันที่[DATE][DATE]67 จนถึงตอนนี้ก็ยังไม่มีเจ้าหน้าที่ติดต่อกลับ และยังไม่เข้าดําเนินการ รบกวนหน่วยงานที่เกี่ยวข้องประสานงานตรวจสอบและติดต่อกลับผู้ใช้ไฟฟ้า //[PERSON] หมายเลขติดต่อกลับ[PHONE][PHONE]66